# Kerala helpdesk: Tier-2 ingestion on Kaggle

Runs `ingest.fetch` → `ingest.web` → `ingest.extract` (OCR) → `ingest.build_docs` → `check_kb` → tests,
and saves everything to `/kaggle/working/ingest_out/` (+ `ingest_out.zip`).

**How to run**
1. Settings → **Internet on** (needs a phone-verified account). Accelerator: none (CPU, 4 cores).
2. Edit the CONFIG cell if needed.
3. **Save Version → Save & Run All (Commit)**. You can close the browser; a run may take up to 12 h.
4. When it finishes: open the version → **Output** → download `ingest_out.zip`, unzip it in your repo root.

**Resume / add more later**: in a new version, **Add Input** → *Your Work* → this notebook's previous version.
Its `ingest_out` is copied back in first, so finished charters and crawled pages are not redone.
To add sources, edit `ingest/sources.csv` / `ingest/web_sources.csv` in GitHub (or use `EXTRA_*` below) and run again.

**If LSGD sites refuse Kaggle's servers** (`FAILED` in the fetch log): download the PDFs on your Mac,
upload them as a Kaggle Dataset named like the slugs (`kuravilangad-gp-2026.pdf`, ...), add it as Input,
and set `RUN_FETCH = False`.

In [ ]:
# ---- CONFIG ----
REPO_URL   = "https://github.com/cksajil/kerala-voice-helpdesk"
BRANCH     = ""       # empty = the repo's default branch
RUN_FETCH  = True     # download charters listed in ingest/sources.csv
RUN_WEB    = True     # crawl sections listed in ingest/web_sources.csv
WEB_MAX    = 30       # pages per website section
RUN_EXTRACT= True     # OCR every PDF not yet extracted
JOBS       = 4        # PDFs in parallel (Kaggle CPU has 4 cores)
BUDGET_MIN = 600      # stop starting new PDFs after this many minutes (Kaggle's limit is 12 h)
RUN_TESTS  = True
# Optional extra rows, appended to the CSVs for this run only (same columns as the files):
EXTRA_SOURCES = [
    # "slug,lsgi_ml,lsgi_en,district,year,url,kb_alias",
]
EXTRA_WEB_SOURCES = [
    # "slug,url,path_prefix,max_pages,notes",
]

In [ ]:
%%bash
set -e
apt-get -qq update > /dev/null
apt-get -qq install -y tesseract-ocr tesseract-ocr-mal > /dev/null
tesseract --list-langs | grep -q '^mal$' && echo "tesseract + Malayalam OK" 

In [ ]:
import os, subprocess, sys
os.chdir("/kaggle/working")
if not os.path.exists("repo"):
    subprocess.run(["git", "clone", "-q", "--depth", "1", *(["-b", BRANCH] if BRANCH else []), REPO_URL, "repo"], check=True)
os.chdir("/kaggle/working/repo")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-ingest.txt"], check=True)
for path, rows in (("ingest/sources.csv", EXTRA_SOURCES), ("ingest/web_sources.csv", EXTRA_WEB_SOURCES)):
    if rows:
        with open(path, "a", encoding="utf-8") as f:
            f.write("\n" + "\n".join(rows) + "\n")
        print(f"added {len(rows)} rows to {path}")
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
# Resume: bring back the previous run's output and any PDFs uploaded as a dataset.
import glob, shutil, zipfile, pathlib
restored = 0
for z in glob.glob("/kaggle/input/**/ingest_out.zip", recursive=True):
    zipfile.ZipFile(z).extractall("."); restored += 1; print("restored", z)
for d in glob.glob("/kaggle/input/**/ingest_out", recursive=True):
    if os.path.isdir(d):
        shutil.copytree(d, ".", dirs_exist_ok=True); restored += 1; print("restored", d)
pathlib.Path("ingest/raw").mkdir(exist_ok=True)
pdfs = [p for p in glob.glob("/kaggle/input/**/*.pdf", recursive=True) if "/ingest_out/" not in p]
for p in pdfs:
    shutil.copy(p, "ingest/raw/")
print(f"{restored} previous outputs restored, {len(pdfs)} uploaded PDFs copied")
print("already extracted:", sorted(x.stem for x in pathlib.Path("ingest/pages").glob("*.jsonl")))

In [ ]:
if RUN_FETCH:
    !python -m ingest.fetch 2>&1 | tee fetch.log
    !grep -E "FAILED|SKIPPED" fetch.log || echo "no failures" 

In [ ]:
if RUN_WEB:
    !python -m ingest.web --max {WEB_MAX} 2>&1 | tail -40

In [ ]:
if RUN_EXTRACT:
    !python -m ingest.extract --jobs {JOBS} --budget {BUDGET_MIN} 2>&1 | grep -v "^  " 

In [ ]:
!python -m ingest.build_docs
!python -m ingest.check_kb || true
if RUN_TESTS:
    !python -m pytest -q 2>&1 | tail -5

In [ ]:
# Save outputs: a folder (for the next version's Input) and a zip (to download).
out = pathlib.Path("/kaggle/working/ingest_out")
shutil.rmtree(out, ignore_errors=True)
keep = ["data/docs.jsonl", "ingest/pages", "ingest/raw/manifest.json", "ingest/raw/web_manifest.json",
        "ingest/report.md", "ingest/kb_check.md", "ingest/changes.md", "ingest/charter_changes.md",
        "ingest/web_pdfs.txt", "fetch.log"]
for k in keep:
    src = pathlib.Path(k)
    if src.is_dir():
        shutil.copytree(src, out / k, ignore=shutil.ignore_patterns("*.partial"))
    elif src.exists():
        (out / k).parent.mkdir(parents=True, exist_ok=True); shutil.copy(src, out / k)
shutil.make_archive("/kaggle/working/ingest_out", "zip", out)
n = sum(1 for _ in open("data/docs.jsonl", encoding="utf-8"))
print(f"{n} passages; saved /kaggle/working/ingest_out.zip")
shutil.rmtree("/kaggle/working/repo/ingest/work", ignore_errors=True)   # page images: big, not needed